In [1]:
import sys
sys.path.append('/home/azureuser/prathyusha/')
from utils import *
spark = instantiate_spark_sedona("50g")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
SLF4J: Failed to load class "org.slf4j.impl.StaticMDCBinder".
SLF4J: Defaulting to no-operation MDCAdapter implementation.
SLF4J: See http://www.slf4j.org/codes.html#no_static_mdc_binder for further details.


Spark Session and SedonaContext have been successfully initiated.


In [2]:
from pyspark.sql.functions import *

final_df = spark.read.parquet("abfss://propheus-data-science@propheusdatabay.dfs.core.windows.net/Thailand/thailand_quadkeys/Quadkey_pop_hh_estimated_updated")
final_df_req = final_df.filter(~col('adm1_name').isin('bangkok', 'nonthaburi', 'pathumthani', 'samutprakan', 'samutsakhon', 'nakhonpathom'))
final_df_req.show()

+----------------+------------+--------------------+------------------+-----------------------+------------------+--------------------+---------------+
|         quadkey|   adm1_name|Pop_nso_to_hdx_ratio|   hdx_Quadkey_Pop|nso_adjusted_population|sum_floor_area_sqm|Qk_pop_to_hh_sampled|Qk_estimated_hh|
+----------------+------------+--------------------+------------------+-----------------------+------------------+--------------------+---------------+
|1322121030100010|amnatcharoen|  1.3095653425533569| 142.0465920000001|     186.01929391101703| 76090.12173170949|   2.449373527632673|           76.0|
|1322121021232330|amnatcharoen|  1.3095653425533569|201.32532000000032|      263.6486616504646| 62566.12192884956|   2.488199875170112|          106.0|
|1322121030102232|amnatcharoen|  1.3095653425533569|135.38815800000015|     177.29963950893818| 54976.98828964075|   2.523829708720109|           70.0|
|1322121021312003|amnatcharoen|  1.3095653425533569|199.33478199999976|      261.0419220

In [3]:
mapping = spark.read.parquet("abfss://propheus-data-science@propheusdatabay.dfs.core.windows.net/Thailand/thailand_quadkeys/thailand_quadkeys_w_subdistrict")
mapping.show()

+----------------+--------------------+--------------------+-----------------+------------+---------+--------------------+
|         quadkey|    quadkey_geometry|    quadkey_centroid|        adm3_name|   adm2_name|adm1_name|            geometry|
+----------------+--------------------+--------------------+-----------------+------------+---------+--------------------+
|1322122012022123|POLYGON ((101.980...|POINT (101.983337...|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|POLYGON ((102.032...|
|1322122012022132|POLYGON ((101.986...|POINT (101.988830...|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|POLYGON ((102.032...|
|1322122012022133|POLYGON ((101.991...|POINT (101.994323...|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|POLYGON ((102.032...|
|1322122012023022|POLYGON ((101.997...|POINT (101.999816...|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|POLYGON ((102.032...|
|1322122012023023|POLYGON ((102.002...|POINT (102.005310...|Thung Mahacharoen|Wang Nam Yen|  Sa Kaeo|POLYGON ((102.032...|
|132212201202302

In [4]:
final_df_joined = final_df_req.select('quadkey', 'nso_adjusted_population', 'Qk_pop_to_hh_sampled').join(
    mapping.select('quadkey', 'adm2_name', 'adm3_name', 'adm1_name'),
    on = 'quadkey',
    how = 'inner'
)
final_df_joined.count()

960076

In [5]:
final_df_joined.show()

+----------------+-----------------------+--------------------+-------------------+------------+------------+
|         quadkey|nso_adjusted_population|Qk_pop_to_hh_sampled|          adm2_name|   adm3_name|   adm1_name|
+----------------+-----------------------+--------------------+-------------------+------------+------------+
|1322010330330112|       3.85783809004375|  3.0752889070291864|         Pang Mapha|   Na Pu Pom|Mae Hong Son|
|1322010330331231|      4.590031058358725|   3.514785880528593|         Pang Mapha|   Na Pu Pom|Mae Hong Son|
|1322010331201320|      2.149387830642142|   3.665291787562955|         Pang Mapha|   Na Pu Pom|Mae Hong Son|
|1322010331201323|     3.1295666306991308|   2.997189388366925|         Pang Mapha|   Na Pu Pom|Mae Hong Son|
|1322010331202200|       9.18006211671745|   2.760167183741254|         Pang Mapha|   Na Pu Pom|Mae Hong Son|
|1322010331202212|      4.360529505440788|     2.9530662261496|         Pang Mapha|   Na Pu Pom|Mae Hong Son|
|132201033

In [6]:
district_level_nso_pop = final_df_joined.groupBy('adm1_name', 'adm2_name').agg(
    sum('nso_adjusted_population').alias('nso_adjusted_hdx_population')
)
district_level_nso_pop.show()

+-------------------+-----------------+---------------------------+
|          adm1_name|        adm2_name|nso_adjusted_hdx_population|
+-------------------+-----------------+---------------------------+
|         Chiang Rai|          Pa Daet|         24578.121527529765|
|                Nan|        Santi Suk|           13083.5321560293|
|     Kamphaeng Phet|  Pang Sila Thong|         27757.451102276405|
|      Maha Sarakham|     Yang Sisurat|         28337.108878189585|
|           Saraburi|         Nong Don|         14834.771568171083|
|   Nong Bua Lam Phu|         Na Klang|          90339.88305616786|
|  Nakhon Ratchasima|      Chum Phuang|          79499.40946119936|
|           Buri Ram|    Non Din Daeng|          19530.01458633106|
|             Roi Et|        Nong Phok|          51058.00226440467|
|   Ubon Ratchathani|      Khong Chiam|         30983.471244773307|
|   Ubon Ratchathani|        Buntharik|           83897.8993857903|
|Nakhon Si Thammarat|      Pak Phanang|         

In [7]:
district_level_nso_pop.count()

848

In [8]:
district_truth_set = spark.read.csv('Population Adjustment - Districts - Final_District_Adjustments.csv', header=True)
district_truth_set = district_truth_set.select('Province', 'District', 'Population - Truth set (nso matched)')
district_truth_set.show()

+-----------------+-------------------+------------------------------------+
|         Province|           District|Population - Truth set (nso matched)|
+-----------------+-------------------+------------------------------------+
|       Udon Thani|          Thung Fon|                         24593.84085|
|Nakhon Ratchasima|              Khong|                         80640.37428|
|         Lop Buri|    Mueang Lop Buri|                         248302.4556|
|        Khon Kaen|          Chum Phae|                         120085.3787|
|      Suphan Buri|            U Thong|                         97047.32322|
|              Nan|             Na Noi|                         31401.04129|
|     Kanchanaburi|          Tha Muang|                         105003.4113|
|      Surat Thani|          Phrasaeng|                         72105.59717|
|          Bangkok|          Lat Phrao|                         246192.3001|
|     Kanchanaburi|           Tha Maka|                         128199.0563|

In [9]:
multiplier_join = district_level_nso_pop.join(
    district_truth_set,
    ((district_level_nso_pop.adm1_name == district_truth_set.Province) &
    (district_level_nso_pop.adm2_name == district_truth_set.District)),
    how = 'inner'
).drop(district_truth_set.Province, district_truth_set.District)

multiplier_join = multiplier_join.withColumn("Multiplier", col('Population - Truth set (nso matched)')/col('nso_adjusted_hdx_population'))
multiplier_join.show()

+-------------------+-----------------+---------------------------+------------------------------------+------------------+
|          adm1_name|        adm2_name|nso_adjusted_hdx_population|Population - Truth set (nso matched)|        Multiplier|
+-------------------+-----------------+---------------------------+------------------------------------+------------------+
|         Chiang Rai|          Pa Daet|         24578.121527529765|                         24026.71372| 0.977565095570378|
|                Nan|        Santi Suk|           13083.5321560293|                         12634.08971|0.9656482331629244|
|     Kamphaeng Phet|  Pang Sila Thong|         27757.451102276405|                         28631.35473| 1.031483569024532|
|      Maha Sarakham|     Yang Sisurat|         28337.108878189585|                         27431.15232|0.9680293228895035|
|           Saraburi|         Nong Don|         14834.771568171083|                         15502.00202|1.0449774672136172|
|   Nong

In [10]:
multiplier_join.count()

848

In [13]:
edge_cases = multiplier_join.filter(col('Multiplier')>2)
edge_cases.show()

+-----------------+-----------------+---------------------------+------------------------------------+-----------------+
|        adm1_name|        adm2_name|nso_adjusted_hdx_population|Population - Truth set (nso matched)|       Multiplier|
+-----------------+-----------------+---------------------------+------------------------------------+-----------------+
|        Khon Kaen|        Wiang Kao|         136.91516442371127|                          26580.6012|194.1392051923557|
|Nakhon Ratchasima|Chaloem Phra Kiat|          9469.327622144923|                         32809.58129|3.464826923220153|
|      Suphan Buri|        Don Chedi|          46262.48569254694|                         202022.3353|4.366871608296364|
|           Rayong|      Pluak Daeng|          67810.26054058631|                         161836.9948| 2.38661514510973|
+-----------------+-----------------+---------------------------+------------------------------------+-----------------+



In [12]:
# multiplier_join_adm1_adm2
# multiplier_join.toPandas().to_csv('multiplier_join_adm1_adm2.csv', index=False)

In [14]:
Matchinh_truth_set = multiplier_join.select('adm1_name', 'adm2_name', 'Multiplier').join(
    edge_cases,
    on = ['adm1_name', 'adm2_name'],
    how = 'left_anti'
).join(
    final_df_joined,
    on = ['adm1_name', 'adm2_name'],
    how = 'inner'
)
Matchinh_truth_set = Matchinh_truth_set.withColumn("Truthset_adjusted_population", col('Multiplier')*col('nso_adjusted_population'))
Matchinh_truth_set = Matchinh_truth_set.withColumn("Truthset_estimated_HH_count", col('Truthset_adjusted_population')/col('Qk_pop_to_hh_sampled'))
Matchinh_truth_set.show()

+---------+------------+------------------+----------------+-----------------------+--------------------+--------------+----------------------------+---------------------------+
|adm1_name|   adm2_name|        Multiplier|         quadkey|nso_adjusted_population|Qk_pop_to_hh_sampled|     adm3_name|Truthset_adjusted_population|Truthset_estimated_HH_count|
+---------+------------+------------------+----------------+-----------------------+--------------------+--------------+----------------------------+---------------------------+
|Bueng Kan|Phon Charoen|1.1313178062654645|1322102132122213|      5.992985136861609|   3.644070569137187|Nong Hua Chang|            6.77997079801581|          1.860548710400281|
|Bueng Kan|Phon Charoen|1.1313178062654645|1322102132123212|      5.778315295107728|   3.681289655698566|Nong Hua Chang|           6.537110983571455|         1.7757665369939395|
|Bueng Kan|Phon Charoen|1.1313178062654645|1322102132132302|      4.770791281866241|   3.642787803275275|     

In [33]:
buildings = spark.read.parquet("abfss://propheus-data-science@propheusdatabay.dfs.core.windows.net/Thailand/Gee_Buildings/buildings_full_dataset_with_quadkey_with_floor_area.parquet")
buildings = buildings.drop('province_en', 'district_en', 'tambon_en').join(
    mapping,
    on = 'quadkey',
    how = 'inner'
)

# buildings.count()
buildings_req = buildings.join(
    edge_cases.select(col('adm1_name'), col('adm2_name')),
    on = ['adm1_name', 'adm2_name'],
    how = 'inner'
)
buildings_req_qkey_grouped = buildings_req.groupby('quadkey').agg(count('*').alias('Buil_in_quadkey'))
buildings_req_qkey_grouped.show()

+----------------+---------------+
|         quadkey|Buil_in_quadkey|
+----------------+---------------+
|1322102233220033|            298|
|1322102233222203|              3|
|1322102232313330|              3|
|1322102232331023|              4|
|1322102232332230|              3|
|1322102232333120|             89|
|1322102233220200|              7|
|1322102232331033|             25|
|1322102232333133|             38|
|1322102232333102|             88|
|1322102233202231|              5|
|1322120010110013|              1|
|1322102232331012|              4|
|1322102232330301|              2|
|1322102233220202|              6|
|1322120010111032|              1|
|1322102232330133|             10|
|1322102232331103|             77|
|1322102232331220|             11|
|1322102232333112|             38|
+----------------+---------------+
only showing top 20 rows



In [44]:
Matchinh_truth_set_edge_cases = multiplier_join.select('adm1_name', 'adm2_name', 'Multiplier').join(
    edge_cases,
    on = ['adm1_name', 'adm2_name'],
    how = 'inner'
).join(
    final_df_joined,
    on = ['adm1_name', 'adm2_name'],
    how = 'inner'
).join(
    buildings_req_qkey_grouped,
    on = 'quadkey',
    how = 'inner'
)
Matchinh_truth_set_edge_cases.show()





+----------------+---------+---------+-----------------+---------------------------+------------------------------------+-----------------+-----------------------+--------------------+--------------------+---------------+
|         quadkey|adm1_name|adm2_name|       Multiplier|nso_adjusted_hdx_population|Population - Truth set (nso matched)|       Multiplier|nso_adjusted_population|Qk_pop_to_hh_sampled|           adm3_name|Buil_in_quadkey|
+----------------+---------+---------+-----------------+---------------------------+------------------------------------+-----------------+-----------------------+--------------------+--------------------+---------------+
|1322102233220033|Khon Kaen|Wiang Kao|194.1392051923557|         136.91516442371127|                          26580.6012|194.1392051923557|                    0.0|   2.407084425025908|            Khao Noi|            298|
|1322102233222203|Khon Kaen|Wiang Kao|194.1392051923557|         136.91516442371127|                          26

In [43]:
Matchinh_truth_set_edge_cases.columns

['quadkey',
 'adm1_name',
 'adm2_name',
 'Multiplier',
 'Population - Truth set (nso matched)',
 'nso_adjusted_hdx_population',
 'Population - Truth set (nso matched)',
 'Multiplier',
 'nso_adjusted_population',
 'Qk_pop_to_hh_sampled',
 'adm3_name',
 'Buil_in_quadkey',
 'Buil_in_district']

In [45]:
from pyspark.sql.window import Window

w = Window.partitionBy('adm1_name', 'adm2_name')

Matchinh_truth_set_edge_cases = Matchinh_truth_set_edge_cases.withColumn(
    "Buil_in_district", count("*").over(w)
)


Matchinh_truth_set_edge_cases = Matchinh_truth_set_edge_cases.withColumn("Truthset_adjusted_population", col('Population - Truth set (nso matched)')*col('Buil_in_quadkey')/col('Buil_in_district'))
Matchinh_truth_set_edge_cases = Matchinh_truth_set_edge_cases.withColumn("Truthset_estimated_HH_count", col('Truthset_adjusted_population')/col('Qk_pop_to_hh_sampled'))

Matchinh_truth_set_edge_cases.show()

+----------------+---------+---------+-----------------+---------------------------+------------------------------------+-----------------+-----------------------+--------------------+--------------------+---------------+----------------+----------------------------+---------------------------+
|         quadkey|adm1_name|adm2_name|       Multiplier|nso_adjusted_hdx_population|Population - Truth set (nso matched)|       Multiplier|nso_adjusted_population|Qk_pop_to_hh_sampled|           adm3_name|Buil_in_quadkey|Buil_in_district|Truthset_adjusted_population|Truthset_estimated_HH_count|
+----------------+---------+---------+-----------------+---------------------------+------------------------------------+-----------------+-----------------------+--------------------+--------------------+---------------+----------------+----------------------------+---------------------------+
|1322102233220033|Khon Kaen|Wiang Kao|194.1392051923557|         136.91516442371127|                          26

In [48]:
Matchinh_truth_set_edge_cases.filter(col('quadkey') == '1322120010111102').show()
Matchinh_truth_set_edge_cases.filter(col('quadkey') == '1322102232332333').show()

+----------------+---------+---------+-----------------+---------------------------+------------------------------------+-----------------+-----------------------+--------------------+--------------------+---------------+----------------+----------------------------+---------------------------+
|         quadkey|adm1_name|adm2_name|       Multiplier|nso_adjusted_hdx_population|Population - Truth set (nso matched)|       Multiplier|nso_adjusted_population|Qk_pop_to_hh_sampled|           adm3_name|Buil_in_quadkey|Buil_in_district|Truthset_adjusted_population|Truthset_estimated_HH_count|
+----------------+---------+---------+-----------------+---------------------------+------------------------------------+-----------------+-----------------------+--------------------+--------------------+---------------+----------------+----------------------------+---------------------------+
|1322120010111102|Khon Kaen|Wiang Kao|194.1392051923557|         136.91516442371127|                          26

+----------------+---------+---------+-----------------+---------------------------+------------------------------------+-----------------+-----------------------+--------------------+--------------------+---------------+----------------+----------------------------+---------------------------+
|         quadkey|adm1_name|adm2_name|       Multiplier|nso_adjusted_hdx_population|Population - Truth set (nso matched)|       Multiplier|nso_adjusted_population|Qk_pop_to_hh_sampled|           adm3_name|Buil_in_quadkey|Buil_in_district|Truthset_adjusted_population|Truthset_estimated_HH_count|
+----------------+---------+---------+-----------------+---------------------------+------------------------------------+-----------------+-----------------------+--------------------+--------------------+---------------+----------------+----------------------------+---------------------------+
|1322102232332333|Khon Kaen|Wiang Kao|194.1392051923557|         136.91516442371127|                          26

In [49]:
Matchinh_truth_set_edge_cases_final = Matchinh_truth_set_edge_cases.select('adm1_name', 'adm2_name', 'adm3_name', 'quadkey', 'Truthset_estimated_HH_count').withColumnRenamed(
    'Truthset_estimated_HH_count' , 'HH_Count'
)
Matchinh_truth_set_edge_cases_final.show()

+---------+---------+--------------------+----------------+------------------+
|adm1_name|adm2_name|           adm3_name|         quadkey|          HH_Count|
+---------+---------+--------------------+----------------+------------------+
|Khon Kaen|Wiang Kao|            Khao Noi|1322102233220033| 8394.670878347904|
|Khon Kaen|Wiang Kao|Mueang Kao Phatthana|1322102233222203| 64.29369639770589|
|Khon Kaen|Wiang Kao|          Nai Mueang|1322102232313330| 64.88704747021397|
|Khon Kaen|Wiang Kao|          Nai Mueang|1322102232331023| 89.42458784019755|
|Khon Kaen|Wiang Kao|Mueang Kao Phatthana|1322102232332230| 64.08438706889328|
|Khon Kaen|Wiang Kao|          Nai Mueang|1322102232333120| 2313.498979736779|
|Khon Kaen|Wiang Kao|          Nai Mueang|1322102233220200| 154.3890905965058|
|Khon Kaen|Wiang Kao|          Nai Mueang|1322102232331033| 623.5420127799686|
|Khon Kaen|Wiang Kao|Mueang Kao Phatthana|1322102232333133| 942.1242169479998|
|Khon Kaen|Wiang Kao|          Nai Mueang|1322102232

In [50]:
Matchinh_truth_set_71 = Matchinh_truth_set.select('adm1_name', 'adm2_name', 'adm3_name', 'quadkey', 'Truthset_estimated_HH_count').withColumnRenamed(
    'Truthset_estimated_HH_count' , 'HH_Count'
).unionByName(Matchinh_truth_set_edge_cases_final)


Matchinh_truth_set_71.show()

+---------+------------+--------------+----------------+------------------+
|adm1_name|   adm2_name|     adm3_name|         quadkey|          HH_Count|
+---------+------------+--------------+----------------+------------------+
|Bueng Kan|Phon Charoen|Nong Hua Chang|1322102132122213| 1.860548710400281|
|Bueng Kan|Phon Charoen|Nong Hua Chang|1322102132123212|1.7757665369939395|
|Bueng Kan|Phon Charoen|     Si Samran|1322102132132302|1.4816347859456862|
|Bueng Kan|Phon Charoen|Nong Hua Chang|1322102132300033|0.7019901663934888|
|Bueng Kan|Phon Charoen|Nong Hua Chang|1322102132300110| 1.002630940546242|
|Bueng Kan|Phon Charoen|Nong Hua Chang|1322102132300111|0.8203529670495187|
|Bueng Kan|Phon Charoen|Nong Hua Chang|1322102132302103| 1.980523082446843|
|Bueng Kan|Phon Charoen|Nong Hua Chang|1322102132302312|  4.27887490383836|
|Bueng Kan|Phon Charoen|     Si Samran|1322102132310130|0.8167462593230094|
|Bueng Kan|Phon Charoen|     Si Samran|1322102132310310|0.6348806416068985|
|Bueng Kan|P

In [51]:
Matchinh_truth_set_71.count()

960076

In [15]:


Matchinh_truth_set_71.filter(col('quadkey') == '1322120010111102').show()

[Stage 120:======================================================>(98 + 1) / 99]

+---------+---------+--------------------+----------------+-----------------+
|adm1_name|adm2_name|           adm3_name|         quadkey|         HH_Count|
+---------+---------+--------------------+----------------+-----------------+
|Khon Kaen|Wiang Kao|Mueang Kao Phatthana|1322120010111102|872.0806118569035|
+---------+---------+--------------------+----------------+-----------------+



In [54]:
Matchinh_truth_set_71.toPandas().to_csv('Matchinh_truth_set_71.csv', index=False)

## Combining all Thailand

In [52]:

def to_snake_case(col):
    return lower(
        regexp_replace(col, r'[^a-zA-Z0-9]+', '_')
    )

In [ ]:
df_5_provinces = spark.read.parquet('/home/azureuser/prathyusha/Kearney/prathyusha/EDA/selected_5_province_HH_estimates.parquet')
df_5_provinces.show()

In [48]:
df_final = Matchinh_truth_set_71.unionByName(df_5_provinces)

df_final = df_final.withColumn('adm3_name', to_snake_case(col('adm3_name')))
df_final = df_final.withColumn('adm2_name', to_snake_case(col('adm2_name')))
df_final = df_final.withColumn('adm1_name', to_snake_case(col('adm1_name')))
df_final.count()

977366

In [49]:
df_final.show()

+---------+------------+-----------+----------------+------------------+
|adm1_name|   adm2_name|  adm3_name|         quadkey|          HH_Count|
+---------+------------+-----------+----------------+------------------+
| buri_ram|lam_plai_mat| khok_sa_at|1322120300123002|0.5958331335966651|
| buri_ram|lam_plai_mat| khok_sa_at|1322120300123300|0.7658522226010662|
| buri_ram|lam_plai_mat|mueang_faek|1322120300131003|1.1668015776724898|
| buri_ram|lam_plai_mat|     bu_pho|1322120300132201| 35.96388227477043|
| buri_ram|lam_plai_mat|     bu_pho|1322120300132230|1.5317887547621196|
| buri_ram|lam_plai_mat|mueang_faek|1322120300133222|1.6409855152091184|
| buri_ram|lam_plai_mat|   hin_khon|1322120300211211| 2.803688438722988|
| buri_ram|lam_plai_mat|phathai_rin|1322120300223322| 4.897439128893392|
| buri_ram|lam_plai_mat|   hin_khon|1322120300231001|13.638250461343143|
| buri_ram|lam_plai_mat|   hin_khon|1322120300231011|22.756942549606958|
| buri_ram|lam_plai_mat|phathai_rin|132212030023232

In [55]:
bangkok = spark.read.parquet("abfss://propheus-data-science@propheusdatabay.dfs.core.windows.net/Thailand/bangkok/quadkey_level_household_estimates/")
bangkok = bangkok.drop('province_en')
bangkok = bangkok.withColumnRenamed('total_estimated_households_adjusted', 'HH_Count')
bangkok.show()

+---------+----------------+---------+---------+--------+
|adm2_name|         quadkey|adm3_name|adm1_name|HH_Count|
+---------+----------------+---------+---------+--------+
|  bang_na|1322033110221100|  bang_na|  bangkok|    1531|
|  bang_na|1322033110203303|  bang_na|  bangkok|     879|
|  bang_na|1322033110203230|  bang_na|  bangkok|    9671|
|  bang_na|1322033110212223|  bang_na|  bangkok|     698|
|  bang_na|1322033110203222|  bang_na|  bangkok|     956|
|  bang_na|1322033110203212|  bang_na|  bangkok|    1181|
|  bang_na|1322033110203232|  bang_na|  bangkok|    2124|
|  bang_na|1322033110203320|  bang_na|  bangkok|     920|
|  bang_na|1322033110221012|  bang_na|  bangkok|    2263|
|  bang_na|1322033110203323|  bang_na|  bangkok|     658|
|  bang_na|1322033110202333|  bang_na|  bangkok|     353|
|  bang_na|1322033110221000|  bang_na|  bangkok|     613|
|  bang_na|1322033110230002|  bang_na|  bangkok|     679|
|  bang_na|1322033110221013|  bang_na|  bangkok|    2582|
|  bang_na|132

In [56]:
df_final_final = df_final.unionByName(bangkok)
df_final_final.count()

981791

In [57]:
df_final_final.select('quadkey', 'adm3_name', 'adm2_name', 'adm1_name').distinct().count()

977153

In [100]:
df_final_final_final = df_final_final.groupBy('quadkey', 'adm3_name', 'adm2_name', 'adm1_name').agg(sum('HH_Count').alias('HH_Count'))
df_final_final_final = df_final_final_final.withColumn('HH_Count',when(col('quadkey') == '1322033110000021',10527).otherwise(col('HH_Count')))
df_final_final_final = df_final_final_final.withColumn('HH_Count',when(col('quadkey') == '1322033110320302',103).otherwise(col('HH_Count')))
df_final_final_final.count()

981418

In [101]:
df_final_final_final.select('quadkey', 'adm3_name', 'adm2_name', 'adm1_name').distinct().count()

981418

In [102]:
df_final_final_final = df_final_final_final.filter(col('HH_Count') > 0)
df_final_final_final = df_final_final_final.withColumn('HH_Count', ceil(col('HH_Count')))
df_final_final_final.toPandas().describe()

,HH_Count
count,981037.000000
mean,26.989265
std,111.061506
min,1.000000
25%,2.000000
50%,5.000000
75%,21.000000
max,11787.000000


In [103]:
df_final_final_final.count()

981037

In [104]:
df_final_final_final.toPandas().to_csv('Thailand_HH_Count.csv', index=False)

In [106]:
df_final_final_final.select(sum('HH_Count')).show()

+-------------+
|sum(HH_Count)|
+-------------+
|     26477468|
+-------------+



In [91]:
df_final_final_final.toPandas()['HH_Count'].describe()

count    981418.000000
mean         26.489837
std         111.054433
min           0.000000
25%           1.520495
50%           4.466675
75%          20.509905
max       11787.000000
Name: HH_Count, dtype: float64

In [92]:
df_final_final_final.select('adm1_name','adm2_name','adm3_name').distinct().count()

7417

In [93]:
df_final_final_final.select(sum('HH_Count')).show()

+--------------------+
|       sum(HH_Count)|
+--------------------+
|2.5997603172088426E7|
+--------------------+



In [94]:
df_final_final_final.toPandas().isna().sum()

quadkey      0
adm3_name    0
adm2_name    0
adm1_name    0
HH_Count     0
dtype: int64

In [61]:
df_final_final_final.show()

+----------------+--------------+--------------+------------+------------------+
|         quadkey|     adm3_name|     adm2_name|   adm1_name|          HH_Count|
+----------------+--------------+--------------+------------+------------------+
|1322120300312333|    thamenchai|  lam_plai_mat|    buri_ram|10.571285846115003|
|1322120300323213| nong_bua_khok|  lam_plai_mat|    buri_ram|0.5925368108237239|
|1322120300300232|      khok_lam|  lam_plai_mat|    buri_ram|0.6238949941687498|
|1322120300132231|        bu_pho|  lam_plai_mat|    buri_ram|1.1184703548448311|
|1322120300301130|     talat_pho|  lam_plai_mat|    buri_ram|0.8338654743656927|
|1322120300313032|    thamenchai|  lam_plai_mat|    buri_ram| 83.55429360407037|
|1322120300123012|    khok_sa_at|  lam_plai_mat|    buri_ram|0.7891076394048853|
|1322120300121313|    khok_sa_at|  lam_plai_mat|    buri_ram|3.9898632074513287|
|1322120302000313|   phathai_rin|  lam_plai_mat|    buri_ram|2.0713947517102063|
|1322120300231132|  lam_plai

In [74]:
df_final_final_final.select('quadkey').distinct().count()

981418

In [ ]:
df_final_final_final = df_final_final_final.filter(col('HH_Count') > 0)
df_final_final_final = df_final_final_final.withColumn('HH_Count', col('HH_Count').cast('integer'))
df_final_final_final = df_final_final_final.filter(col('HH_Count') > 0)
df_final_final_final.toPandas().describe()

,HH_Count
count,845295.000000
mean,30.193314
std,121.287899
min,1.000000
25%,2.000000
50%,6.000000
75%,25.000000
max,19844.000000


In [78]:
df_final_final_final.orderBy(col('HH_Count').desc()).show()

+----------------+-------------------+-------------------+------------+--------+
|         quadkey|          adm3_name|          adm2_name|   adm1_name|HH_Count|
+----------------+-------------------+-------------------+------------+--------+
|1322033110000021|            ban_mai|           pak_kret|  nonthaburi|   19844|
|1322033110320302|       bang_chalong|          bang_phli|samut_prakan|   12345|
|1322033110022233|        huai_khwang|        huai_khwang|     bangkok|   11787|
|1322033110201222|       phra_khanong|        khlong_toei|     bangkok|   10176|
|1322033101310233|        dao_khanong|          thon_buri|     bangkok|    9891|
|1322033110000001|            ban_mai|           pak_kret|  nonthaburi|    9867|
|1322033110203230|            bang_na|            bang_na|     bangkok|    9671|
|1322031332201302|       khlong_nueng|       khlong_luang|pathum_thani|    9075|
|1322033101310231|         talat_phlu|          thon_buri|     bangkok|    8894|
|1322031323322213|     bang_

In [80]:
from utils import *

In [82]:
@udf(StringType())
def quadkey_to_wkt_geometry(qk):
    import mercantile
    from shapely.geometry import shape
    if not qk:
        return None
    try:

        tile_feature = mercantile.feature(mercantile.quadkey_to_tile(qk))
        return shape(tile_feature['geometry']).wkt
    except:
        return None

In [83]:
df_final_final_final = df_final_final_final.withColumn('quadkey_geometry', quadkey_to_wkt_geometry('quadkey'))
df_final_final_final.show()

+----------------+--------------+--------------+------------+--------+--------------------+
|         quadkey|     adm3_name|     adm2_name|   adm1_name|HH_Count|    quadkey_geometry|
+----------------+--------------+--------------+------------+--------+--------------------+
|1322120300312333|    thamenchai|  lam_plai_mat|    buri_ram|      10|POLYGON ((102.958...|
|1322120300132231|        bu_pho|  lam_plai_mat|    buri_ram|       1|POLYGON ((102.936...|
|1322120300313032|    thamenchai|  lam_plai_mat|    buri_ram|      83|POLYGON ((102.974...|
|1322120300121313|    khok_sa_at|  lam_plai_mat|    buri_ram|       3|POLYGON ((102.914...|
|1322120302000313|   phathai_rin|  lam_plai_mat|    buri_ram|       2|POLYGON ((102.694...|
|1322120300231132|  lam_plai_mat|  lam_plai_mat|    buri_ram|      55|POLYGON ((102.821...|
|1322120300310312|   mueang_faek|  lam_plai_mat|    buri_ram|       1|POLYGON ((102.952...|
|1322120302101003|    khok_klang|  lam_plai_mat|    buri_ram|      56|POLYGON ((

In [84]:
df_final_final_final.orderBy(col('HH_Count').desc()).show()

+----------------+-------------------+-------------------+------------+--------+--------------------+
|         quadkey|          adm3_name|          adm2_name|   adm1_name|HH_Count|    quadkey_geometry|
+----------------+-------------------+-------------------+------------+--------+--------------------+
|1322033110000021|            ban_mai|           pak_kret|  nonthaburi|   19844|POLYGON ((100.552...|
|1322033110320302|       bang_chalong|          bang_phli|samut_prakan|   12345|POLYGON ((100.744...|
|1322033110022233|        huai_khwang|        huai_khwang|     bangkok|   11787|POLYGON ((100.563...|
|1322033110201222|       phra_khanong|        khlong_toei|     bangkok|   10176|POLYGON ((100.590...|
|1322033101310233|        dao_khanong|          thon_buri|     bangkok|    9891|POLYGON ((100.475...|
|1322033110000001|            ban_mai|           pak_kret|  nonthaburi|    9867|POLYGON ((100.552...|
|1322033110203230|            bang_na|            bang_na|     bangkok|    9671|PO

In [ ]:
df_final_final_final.filter(col('HH_Count'))

## Validations

In [11]:
District_HH_count = Matchinh_truth_set.groupBy('adm1_name', 'adm2_name').agg(
    sum('Truthset_estimated_HH_count').alias('Truthset_estimated_HH_count')
)
District_HH_count.show()

+-------------------+-----------------+---------------------------+
|          adm1_name|        adm2_name|Truthset_estimated_HH_count|
+-------------------+-----------------+---------------------------+
|          Bueng Kan|     Phon Charoen|          12457.64846471362|
|           Buri Ram|          Krasang|         24955.945337244244|
|           Buri Ram|     Lam Plai Mat|         32837.496833696394|
|           Buri Ram|    Non Din Daeng|          8923.438505503214|
|       Chachoengsao|   Bang Nam Priao|         38398.099432736635|
|       Chachoengsao|       Plaeng Yao|         23142.965758187805|
|         Chiang Rai|        Doi Luang|          7229.027072022763|
|         Chiang Rai|     Mae Fa Luang|          27526.35061722876|
|         Chiang Rai|          Pa Daet|          9195.349528808569|
|            Kalasin|         Don Chan|          5217.434431561696|
|            Kalasin|       Yang Talat|          37326.48097030411|
|     Kamphaeng Phet|  Pang Sila Thong|         

In [12]:
estm_province_hh = District_HH_count.groupBy('adm1_name').agg(sum('Truthset_estimated_HH_count').alias('Truthset_estimated_HH_count'))
estm_province_hh.show()

+-------------------+---------------------------+
|          adm1_name|Truthset_estimated_HH_count|
+-------------------+---------------------------+
|       Chachoengsao|          354958.4279098904|
|        Phitsanulok|         381686.96170950506|
|         Narathiwat|          197925.3528553552|
|         Phetchabun|          335624.7126000605|
|     Kamphaeng Phet|          287030.1425312369|
|      Maha Sarakham|          309884.0630748085|
|                Tak|         189426.87162253424|
|Nakhon Si Thammarat|         523908.58598581306|
|        Phatthalung|         175160.33220115537|
|          Khon Kaen|           678663.371119605|
|             Roi Et|         367345.89397383056|
|           Lop Buri|         307311.81893243134|
|         Chiang Rai|          430866.6011265144|
|               Trat|         110040.22852363532|
|          Sukhothai|         244251.83209032303|
|        Surat Thani|           429399.212832271|
|   Ubon Ratchathani|          605607.6498995676|


In [14]:
nso_hh_count = spark.read.csv('/home/azureuser/prathyusha/Kearney/prathyusha/EDA/pop_hhcount_nso_w_geom.csv', header=True).select(
    'Province', '2022_HH_count'
)
nso_hh_count.show()

+--------------------+-------------+
|            Province|2022_HH_count|
+--------------------+-------------+
|             bangkok|      3369140|
|         samutprakan|       924779|
|          nonthaburi|       608941|
|         pathumthani|       675573|
|phranakhonsiayutt...|       311101|
|            angthong|        82287|
|             lopburi|       266958|
|            singburi|        62131|
|             chainat|       101490|
|            saraburi|       253998|
|            chonburi|       690875|
|              rayong|       401393|
|         chanthaburi|       190627|
|                trat|        97383|
|        chachoengsao|       318295|
|         prachinburi|       249575|
|         nakhonnayok|        88356|
|              sakaeo|       212976|
|          ratchaburi|       250405|
|        kanchanaburi|       260639|
+--------------------+-------------+
only showing top 20 rows



In [15]:
final_province_check = estm_province_hh.join(
    nso_hh_count,
    regexp_replace(lower(estm_province_hh.adm1_name), " ", "") == nso_hh_count.Province,
    how = 'inner'
)
final_province_check.count()

71

In [ ]:
final_province_check_pd = final_province_check.toPandas()


,adm1_name,Truthset_estimated_HH_count,Province,2022_HH_count
0,Chachoengsao,354958.427910,chachoengsao,318295
1,Phitsanulok,381686.961710,phitsanulok,323784
2,Narathiwat,197925.352855,narathiwat,181888
3,Phetchabun,335624.712600,phetchabun,292360
4,Kamphaeng Phet,287030.142531,kamphaengphet,253396
...,...,...,...,...
66,Mukdahan,150510.762869,mukdahan,132432
67,Mae Hong Son,95524.941337,maehongson,81739
68,Nakhon Phanom,200542.783051,nakhonphanom,173917
69,Krabi,147395.817389,krabi,132475


In [18]:
final_province_check_pd.dtypes

adm1_name                       object
Truthset_estimated_HH_count    float64
2022_HH_count                   object
dtype: object

In [20]:
# final_province_check_pd = final_province_check_pd.drop(columns={'Province'})
final_province_check_pd['2022_HH_count'] = final_province_check_pd['2022_HH_count'].astype(int)
final_province_check_pd['%_diff'] = (final_province_check_pd['Truthset_estimated_HH_count'] - final_province_check_pd['2022_HH_count']) * 100/final_province_check_pd['2022_HH_count']
final_province_check_pd

,adm1_name,Truthset_estimated_HH_count,2022_HH_count,%_diff
0,Chachoengsao,354958.427910,318295,11.518694
1,Phitsanulok,381686.961710,323784,17.883207
2,Narathiwat,197925.352855,181888,8.817158
3,Phetchabun,335624.712600,292360,14.798438
4,Kamphaeng Phet,287030.142531,253396,13.273352
...,...,...,...,...
66,Mukdahan,150510.762869,132432,13.651355
67,Mae Hong Son,95524.941337,81739,16.865806
68,Nakhon Phanom,200542.783051,173917,15.309477
69,Krabi,147395.817389,132475,11.263119


In [21]:
final_province_check_pd['%_diff'].describe()

count    71.000000
mean     13.599212
std       2.592892
min       7.392183
25%      11.844447
50%      13.563154
75%      14.902295
max      21.086709
Name: %_diff, dtype: float64